In [56]:
import torch.nn as nn
from transformers import AutoModelForTokenClassification

In [57]:
import torch

In [58]:
class QASpanDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2)

    def forward(self, data):
        outputs = self.model(input_ids=data['input_ids'],
                             attention_mask=data['attention_mask'])
        return outputs

In [59]:
model = QASpanDetector()

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-u

In [60]:
import pickle

with open('data/debug_preprocessed.pkl', 'rb') as f:
    train_encodings = pickle.load(f)

In [61]:
len(train_encodings['input_ids'])

753

In [62]:
len(train_encodings['input_ids'][0])

451

In [63]:
from preprocess import SquadDataset

train_dataset = SquadDataset(train_encodings)

In [94]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=False)

In [65]:
import torch.nn.functional as F

def get_loss(reg_pred, obj_pred, reg_target, obj_target, answer_start):
    reg_pred = reg_pred[:, answer_start].exp()
    reg_target = reg_target[:, answer_start]

    reg_loss = F.mse_loss(reg_pred, reg_target)
    obj_loss = F.binary_cross_entropy_with_logits(obj_pred, obj_target)

    return reg_loss + obj_loss
    

In [66]:
MODEL_MAX_LENGTH = 512

In [72]:
from torch.optim import AdamW

torch.manual_seed(0)
epochs = 3
optim = AdamW(model.parameters(), lr=5e-5)
for epoch in range(epochs):
    # Set model in train mode
    model.train()
    loss_of_epoch = 0

    print("############Train############")
    for batch_idx, batch in enumerate(train_loader):
        sentence_length = batch['input_ids'].size(1)

        answer_start = int(batch['start_positions'])
        attention_mask = batch['attention_mask']
        attention_mask = F.pad(attention_mask, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)

        reg_target = batch['labels'][:, :, 1]
        obj_target = batch['labels'][:, :, 0]
        obj_target = obj_target.type(torch.FloatTensor)
        reg_target = reg_target.type(torch.FloatTensor)

        outputs = model(batch)

        reg_predict = outputs['logits'][:, :, 1]
        obj_predict = outputs['logits'][:, :, 0]
        reg_predict = F.pad(reg_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)
        obj_predict = F.pad(obj_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)
        obj_predict = obj_predict * attention_mask
        
        loss = get_loss(reg_predict, obj_predict, reg_target, obj_target, answer_start)
        print(loss)
        # loss.backward()
        # optim.step()
        break
    break

############Train############
tensor(3.4504, grad_fn=<AddBackward0>)


In [85]:
a = torch.empty(2, 10)
a

tensor([[1.0469e-38, 1.0653e-38, 1.0194e-38, 2.9389e-39, 1.0194e-38, 9.9184e-39,
         2.9389e-39, 1.0194e-38, 2.9389e-39, 9.2755e-39],
        [9.0918e-39, 1.0010e-38, 9.9184e-39, 1.0653e-38, 9.1837e-39, 9.6428e-39,
         1.0010e-38, 9.1837e-39, 8.9082e-39, 9.2755e-39]])

In [86]:
torch.gather(a, 1, torch.LongTensor([[0], [1]]))

tensor([[1.0469e-38],
        [1.0010e-38]])

In [87]:
b = torch.LongTensor([[0], [1]])
b.size()

torch.Size([2, 1])

In [95]:
test_batch = next(iter(train_loader))

In [96]:
t = test_batch['start_positions']

In [97]:
t

tensor([67, 55])

In [104]:
t.dtype

torch.int64

In [99]:
t = t.reshape(2, 1)

In [101]:
t.size()

torch.Size([2, 1])

In [100]:
t

tensor([[67],
        [55]])

In [102]:
torch.gather(test_batch['labels'][:, :, 1], 1, t)

tensor([[4.],
        [3.]], dtype=torch.float64)

In [103]:
test_batch['labels'][:, :, 1]

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], dtype=torch.float64)